In [1]:
import pandas as pd

df = pd.read_csv('../data/telco_churn.csv')
print(df.shape)
df.head()


(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].isna().sum())


11


In [4]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())


Churn
0    5174
1    1869
Name: count, dtype: int64


In [5]:
contract_churn = df.groupby('Contract')['Churn'].mean().reset_index()
contract_churn['ChurnRate'] = (contract_churn['Churn'] * 100).round(1)
print(contract_churn)


         Contract     Churn  ChurnRate
0  Month-to-month  0.427097       42.7
1        One year  0.112695       11.3
2        Two year  0.028319        2.8


In [6]:
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'])

tenure_churn = df.groupby('TenureGroup', observed=True)['Churn'].mean().reset_index()
tenure_churn['ChurnRate'] = (tenure_churn['Churn'] * 100).round(1)
print(tenure_churn)


    TenureGroup     Churn  ChurnRate
0   0-12 months  0.476782       47.7
1  13-24 months  0.287109       28.7
2  25-48 months  0.203890       20.4
3  49-72 months  0.095132        9.5


In [8]:
revenue = df.groupby('Churn').agg(
    CustomerCount=('customerID', 'count'),
    AvgMonthlyCharges=('MonthlyCharges', 'mean'),
    TotalMonthlyRevenue=('MonthlyCharges', 'sum')
).reset_index()

revenue['Churn'] = revenue['Churn'].map({1: 'Churned', 0: 'Retained'})
revenue['AvgMonthlyCharges'] = revenue['AvgMonthlyCharges'].round(2)
revenue['TotalMonthlyRevenue'] = revenue['TotalMonthlyRevenue'].round(2)
print(revenue)


      Churn  CustomerCount  AvgMonthlyCharges  TotalMonthlyRevenue
0  Retained           5174              61.27            316985.75
1   Churned           1869              74.44            139130.85


In [9]:
internet_churn = df.groupby('InternetService')['Churn'].mean().reset_index()
internet_churn['ChurnRate'] = (internet_churn['Churn'] * 100).round(1)
print(internet_churn)


  InternetService     Churn  ChurnRate
0             DSL  0.189591       19.0
1     Fiber optic  0.418928       41.9
2              No  0.074050        7.4


In [10]:
def churn_risk_score(row):
    score = 0
    
    if row['Contract'] == 'Month-to-month':
        score += 1
    
    if row['TenureGroup'] == '0-12 months':
        score += 1
    
    if row['InternetService'] == 'Fiber optic':
        score += 1
    
    return score

df['RiskScore'] = df.apply(churn_risk_score, axis=1)
print(df['RiskScore'].value_counts().sort_index())


RiskScore
0    2026
1    1804
2    2297
3     916
Name: count, dtype: int64


In [11]:
risk_churn = df.groupby('RiskScore')['Churn'].mean().reset_index()
risk_churn['ChurnRate'] = (risk_churn['Churn'] * 100).round(1)
print(risk_churn)


   RiskScore     Churn  ChurnRate
0          0  0.033564        3.4
1          1  0.141907       14.2
2          2  0.392686       39.3
3          3  0.701965       70.2


In [12]:
df.to_csv('../exports/telco_churn_clean.csv', index=False)
print("Saved!")


Saved!


In [13]:
# Export 1 — full clean dataset for Tableau
df.to_csv('../exports/telco_churn_clean.csv', index=False)

# Export 2 — summary by risk score
risk_summary = df.groupby('RiskScore').agg(
    CustomerCount=('customerID', 'count'),
    ChurnRate=('Churn', 'mean'),
    AvgMonthlyCharges=('MonthlyCharges', 'mean'),
    TotalMonthlyRevenue=('MonthlyCharges', 'sum')
).reset_index()

risk_summary['ChurnRate'] = (risk_summary['ChurnRate'] * 100).round(1)
risk_summary['AvgMonthlyCharges'] = risk_summary['AvgMonthlyCharges'].round(2)
risk_summary['TotalMonthlyRevenue'] = risk_summary['TotalMonthlyRevenue'].round(2)

risk_summary.to_csv('../exports/risk_score_summary.csv', index=False)
print(risk_summary)
print("\nBoth files saved!")


   RiskScore  CustomerCount  ChurnRate  AvgMonthlyCharges  TotalMonthlyRevenue
0          0           2026        3.4              47.04             95296.80
1          1           1804       14.2              74.32            134070.05
2          2           2297       39.3              65.98            151566.60
3          3            916       70.2              82.08             75183.15

Both files saved!
